In [1]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.optimize import minimize_scalar
import sympy as sp

In [2]:
# Définition des symboles et de la fonction
x, y = sp.symbols('x y')
f = x**2 - y**2
# Affichage
f

x**2 - y**2

In [14]:
# === Gradient symbolique ===
grad_f = sp.Matrix([sp.diff(f, x), sp.diff(f, y)])
print("\nGradient de f :")
grad_f


Gradient de f :


Matrix([
[ 2*x],
[-2*y]])

In [4]:
f_np = sp.lambdify((x, y), f, 'numpy')
grad_np = sp.lambdify((x, y), grad_f, 'numpy')

In [15]:
# === Descente de gradient à pas fixe ===
def gradient_fixed(f, grad, x0, step=0.001, epsilon=1e-6, max_iter=10000):
    xk = np.array(x0, dtype=float)
    traj = [xk.copy()]
    for k in range(max_iter):
        gk = np.array(grad(xk[0], xk[1]), dtype=float).reshape(-1)
        if np.linalg.norm(gk) < epsilon:
            break
        xk = xk - step * gk
        traj.append(xk.copy())
    return np.array(traj)


In [16]:

# === Descente de gradient à pas optimal ===
def gradient_optimal(f, grad, x0, epsilon=1e-6, max_iter=10000):
    xk = np.array(x0, dtype=float)
    traj = [xk.copy()]
    for k in range(max_iter):
        gk = np.array(grad(xk[0], xk[1]), dtype=float).reshape(-1)
        if np.linalg.norm(gk) < epsilon:
            break
        
        # Recherche du pas optimal le long de dk = -grad
        dk = -gk
        phi = lambda s: f(xk[0] + s*dk[0], xk[1] + s*dk[1])
        res = minimize_scalar(phi, bounds=(0, 1), method='bounded')
        sk = res.x
        
        xk = xk + sk * dk
        traj.append(xk.copy())
    return np.array(traj)


In [17]:
# === Points initiaux ===
x0 = [-1.5, 1.5]

traj_fixed = gradient_fixed(f_np, grad_np, x0, step=0.00001)
traj_optimal = gradient_optimal(f_np, grad_np, x0)



/tmp/ipykernel_39979/3528767363.py:12: RuntimeWarning: overflow encountered in scalar add
  phi = lambda s: f(xk[0] + s*dk[0], xk[1] + s*dk[1])
/tmp/ipykernel_39979/3528767363.py:16: RuntimeWarning: overflow encountered in add
  xk = xk + sk * dk


Cette fonction est une forme quadratique non convexe (une selle)
Dans la méthode de pas optimal, on cherche le pas qui minimise la pente

Mais le long de certaines directions, la fonction augmente indéfiniment :

Si tu pars dans la direction où la pente est négative (ici y), f(xk + s*dk) → −∞ quand s augmente.

scipy.optimize.minimize_scalar essaie de trouver un minimum borné, mais la fonction n’a pas de minimum le long de cette direction → résultat : overflow / valeurs énormes.

In [18]:
# === Tracé 3D ===
X = np.linspace(-2, 2, 200)
Y = np.linspace(-1, 3, 200)
X, Y = np.meshgrid(X, Y)
Z = f_np(X, Y)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z, alpha=0.5, cmap='viridis')

# Trajectoires
ax.plot(traj_fixed[:,0], traj_fixed[:,1], f_np(traj_fixed[:,0], traj_fixed[:,1]),
        color='red', marker='o', label='Pas fixe')
ax.plot(traj_optimal[:,0], traj_optimal[:,1], f_np(traj_optimal[:,0], traj_optimal[:,1]),
        color='blue', marker='^', label='Pas optimal')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('f(x,y)')
ax.set_title('Descente de gradient : pas fixe vs pas optimal')
ax.legend()
plt.show()


ValueError: arange: cannot compute length

<Figure size 1200x800 with 1 Axes>